In [1]:
import gzip
import pandas as pd
import pysam

In [5]:
# 1. 定义图片中的 6 个关键 SNP 位点信息 (1-based 坐标)
# 注意：染色体名称根据 VCF 实际格式，可能是 'Chr5', 'chr05', '5' 或 'GWHDRAM00000005'
TARGET_SITES = {
    28505479: {"Hap1": "G", "Hap2": "A", "weight": 1},
    28500881: {"Hap1": "T", "Hap2": "A", "weight": 1},
    28500882: {"Hap1": "T", "Hap2": "C", "weight": 1},
    28500857: {"Hap1": "C", "Hap2": "G", "weight": 1},
    28500854: {"Hap1": "G", "Hap2": "T", "weight": 1},
    28500441: {
        "Hap1": "C",
        "Hap2": "A",
        "weight": 2,
    },  # 位点6 (外显子非同义突变) 设为核心位点
}

VCF_PATH = "/mnt/rice/data/Lianguang_shang/251.SV.vcf.sorted.bgz"


def get_vcf_chrom_name(vcf_obj, candidate_names=["Chr5", "chr05", "5"]):
    """检查 VCF 中的 Chr5 染色体命名格式"""
    contigs = list(vcf_obj.header.contigs)
    for chrom in candidate_names:
        if chrom in contigs:
            return chrom
    # 如果没有找到常见命名，默认返回列表中的第一个 chrom
    return contigs[0] if contigs else 'Chr5'


def parse_vcf_haplotypes(vcf_path, target_sites):
    vcf = pysam.VariantFile(vcf_path)
    chrom = get_vcf_chrom_name(vcf)
    samples = list(vcf.header.samples)

    # 存储每个位点上每个样本的匹配得分
    # score: +1 (Hap1纯合), -1 (Hap2纯合), 0 (杂合/缺失)
    sample_scores = {s: {"Hap1_count": 0, "Hap2_count": 0, "het_count": 0, "missing_count": 0} for s in samples}
    site_genotypes = {s: {} for s in samples}

    # 遍历每个目标位点
    for pos, alleles in target_sites.items():
        # pysam fetch 使用 0-based 坐标半开区间 [start, end)
        records = list(vcf.fetch(chrom, pos - 1, pos))

        if not records:
            print(f"警告: 在 VCF 中未找到位置 {chrom}:{pos} 的变异位点！")
            for s in samples:
                sample_scores[s]["missing_count"] += 1
            continue

        rec = records[0]
        ref = rec.ref
        alts = rec.alts if rec.alts else []
        all_alleles = [ref] + list(alts)

        for s in samples:
            gt = rec.samples[s].allele_indices
            if None in gt or len(gt) == 0:
                sample_scores[s]["missing_count"] += 1
                site_genotypes[s][pos] = "./."
                continue

            # 提取该样本在该位点的实际碱基
            sample_alleles = sorted([all_alleles[idx] for idx in gt])
            gt_str = "/".join(sample_alleles)
            site_genotypes[s][pos] = gt_str

            h1_base = alleles["Hap1"]
            h2_base = alleles["Hap2"]

            # 判断基因型归属
            if sample_alleles[0] == h1_base and sample_alleles[1] == h1_base:
                sample_scores[s]["Hap1_count"] += 1
            elif sample_alleles[0] == h2_base and sample_alleles[1] == h2_base:
                sample_scores[s]["Hap2_count"] += 1
            elif h1_base in sample_alleles and h2_base in sample_alleles:
                sample_scores[s]["het_count"] += 1
            else:
                sample_scores[s]["missing_count"] += 1

    vcf.close()

    # 2. 判别个体 Hap1 / Hap2 / uncertain
    results = []
    total_sites = len(target_sites)

    for s in samples:
        h1_cnt = sample_scores[s]["Hap1_count"]
        h2_cnt = sample_scores[s]["Hap2_count"]
        het_cnt = sample_scores[s]["het_count"]
        site_6_gt = site_genotypes[s].get(28500441, "./.")

        # 分型判定逻辑
        # 条件 1: 关键位点 28500441 必须匹配，且多数位点一致
        if site_6_gt == "C/C" and h1_cnt >= 4 and h2_cnt == 0:
            haptype = "Hap1"
        elif site_6_gt == "A/A" and h2_cnt >= 4 and h1_cnt == 0:
            haptype = "Hap2"
        # 条件 2: 严格纯合判定 (如果位点6无缺失)
        elif h1_cnt > 0 and h2_cnt == 0 and het_cnt == 0:
            haptype = "Hap1"
        elif h2_cnt > 0 and h1_cnt == 0 and het_cnt == 0:
            haptype = "Hap2"
        else:
            haptype = "uncertain"

        res_entry = {
            "Sample": s,
            "Haplotype": haptype,
            "Hap1_Sites_Match": h1_cnt,
            "Hap2_Sites_Match": h2_cnt,
            "Het_Sites": het_cnt,
            "Site6_28500441_GT": site_6_gt,
        }
        # 追加各个位点的具体基因型
        for pos in sorted(target_sites.keys()):
            res_entry[f"Pos_{pos}"] = site_genotypes[s].get(pos, "./.")

        results.append(res_entry)

    df_res = pd.DataFrame(results)
    return df_res


# 运行分析
df_haplo = parse_vcf_haplotypes(VCF_PATH, TARGET_SITES)

# 输出统计分布
print("=== 单倍型分型统计分布 ===")
print(df_haplo["Haplotype"].value_counts())

print("\n=== 分型结果预览 ===")
print(
    df_haplo[
        [
            "Sample",
            "Haplotype",
            "Site6_28500441_GT",
            "Hap1_Sites_Match",
            "Hap2_Sites_Match",
        ]
    ].head(15)
)

# 导出为 CSV 表格
df_haplo.to_csv("251_samples_haplotype_classification.csv", index=False)

[W::tbx_parse1] VCF INFO/END=450125 is smaller than POS at Chr5:972868
This tag will be ignored. Note: only one invalid END tag will be reported.


=== 单倍型分型统计分布 ===
Haplotype
uncertain    251
Name: count, dtype: int64

=== 分型结果预览 ===
   Sample  Haplotype   Site6_28500441_GT  Hap1_Sites_Match  Hap2_Sites_Match
0   NH001  uncertain                 ./.                 0                 0
1   NH002  uncertain                 ./.                 0                 0
2   NH003  uncertain                 ./.                 0                 0
3   NH005  uncertain                 ./.                 0                 0
4   NH006  uncertain                 ./.                 0                 0
5   NH007  uncertain                 ./.                 0                 0
6   NH008  uncertain                 ./.                 0                 0
7   NH009  uncertain                 ./.                 0                 0
8   NH010  uncertain                 ./.                 0                 0
9   NH011  uncertain  N/N[Chr6:30615937[                 0                 0
10  NH012  uncertain                 ./.                 0        